<table>
  <tr>
    <td><div align="left"><font size="30">Machine Vision Toolbox for Python</font></div></td>
    <td><img src="support/figs/VisionToolboxLogo_NoBackgnd@2x.png" width="200"></td>
  </tr>
</table>

<p></p>
<div align="center" style="font-size: 1.5em;">🧑‍💻 OpenCV for humans</div>
<p></p>


(c) Peter Corke 2026

In [ ]:
import importlib
import setup_tutorial
importlib.reload(setup_tutorial)
await setup_tutorial.setup_tutorial(required_toolboxes=['machinevisiontoolbox'], required_packages=['opencv','matplotlib', 'numpy', 'scipy'])

print("\nmachinevision toolbox version:", importlib.import_module("machinevisiontoolbox").__version__)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
np.set_printoptions(linewidth=100, threshold=100, formatter={'float': lambda x: f"{x:8.3g}" if abs(x) > 1e-10 else f"{0:8.3g}"})

# The Image class

Unlike OpenCV, this Toolbox considers images as objects, not generic NumPy arrays. This has huge advantages as I hope you will see.

To read a color image into a Python object is simply

In [ ]:
from machinevisiontoolbox import Image

image = Image.Read("flowers4.png")

and the result is an image container, a Numpy array holding the pixels encapsulated by an object that includes metadata and a boatload of methods and properties.

We can see a lot of pertinant information about the image by

In [ ]:
print(image)

which says that the image contains 3 "color planes" named R, G and B, and each plane is 640x426.

And, we can display it as an image

In [ ]:
image.disp();

🐛 **In the JupyterLite environment the pixel value browsing function doesn't work**.

In [ ]:
image.size

that is, it is 640x426.

We can pull out the NumPy array that holds the pixels

In [ ]:
image.array

which is simply a big 3-dimensional array of 8-bit unsigned integers.

To make this clearer, we can access the value of the pixel at image coordinate (516,351), remember that's (horizontal, vertical) coordinate

In [ ]:
image[516,351]

which is a 3-element 1-dimensional array with `uint8` values.  This is the intensity of red, green and blue respectively.


## Color planes

We can also think of the color image as a stack of three greyscale images that each represent the amoung of redness, greenness and blueness (effectively what the scene looks like through a red, green or blue filter).

Note that we can have images with 2, 3, 4 or more planes.  Hyperspectral cameras can "see" uptp 20 colors (often called bands, short for spectral bands).  The number of planes in the image is given by

In [ ]:
image.nplanes

Some images might have 3 planes but they represent hue, saturation and intensity, rather than red, green and blue. The names of the planes, and which "layer" in the 3D-array they are in is given by 

In [ ]:
image.colororder


Let's let more closely at one of these color planes -- the red plane.

In [ ]:
red = image.red()
red

and we see that is a greyscale image.  The intensity of each pixel is the amount of red at the corresponding pixel in the original color image.

In [ ]:
red.disp()

We could also have written `image.plane(0)`, `image.plane("R")`, or `image[0]`.

## Greyscale and color conversion

We can convert a color image to a greyscale image

In [ ]:
image.mono().disp()

where the greyscale value is a weighted average of the red, greend and blue values.  The hightest weighting is for green and the lowest weighting is for blue, to match the response of the human eye.

We could also colorize the red plane image from above. Here we say that the red value is equal to the corresponding greyscale value, while green and blue are set to zero.  The result is a color image, but the only color is red.

In [ ]:
red.colorize([1,0,0]).disp()

## Histograms

Often we are interested to know the distribution of the pixel values in each plane

In [ ]:
image.stats

which shows that the pixels span the full range from 0 to 255.  A histogram provides more nuanced information

In [ ]:
hist = image.hist()
hist

In [ ]:
hist.plot()

# Standard image processing operations

## Monadic image operations


We will work with a greyscale version of the Mona Lisa image

In [ ]:
mona = Image.Read("monalisa.png", grey=True)


In [ ]:
from machinevisiontoolbox.base import idisp

idisp(mona.array)

First, we display some simple statistics on the minimum and maximum pixel values

In [ ]:
mona.stats

and we see that the average pixel value is well below the mid value of 128.

To get a more nuanced idea of pixel value distribution we will compute a histogram

In [ ]:
histo = mona.hist()
histo

and plot it

In [ ]:
histo.plot()

We can attempt to brighten the image by multiplying all the pixel values by 1.5.  We need to be careful that the values don't exceed the range of the `uint8` type.

In [ ]:
mona.apply(lambda x: np.uint8(np.clip(x*1.5, 0, 255))).disp(vrange=[0,255])

A simple art effect is posterization where we quantise the pixel values

In [ ]:
(mona // 64 * 64).disp(vrange=[0,255])

And we can apply a threshold, testing whether or not each pixel value exceeds 100. The resulting image has pixel values which are either True or False

In [ ]:
(mona > 100).disp()

## Green screening

This is a very common technique in video production, and these days even with video conferencing tools.  We load our foreground image, the one we'd like to superimpose on the background.  In a video this would be the presenter.

In [ ]:
foreground = Image.Read('greenscreen.png', dtype='float')
foreground.disp()

Now we need to determine which pixels are green, the background to be discarded.  We gamma decode the image and compute the chromacity of every pixel $r=R/(R+G+B), g=G/(R+G+B)$.  The result is an image with two planes

In [ ]:
cc = foreground.gamma_decode('sRGB').chromaticity()
print(cc)

We will apply a threshold to $g$ but we need to understand the distribution before we can choose the threshold value.  For that we'll compute and display a histogram.

In [ ]:
h = cc.plane('g').hist()
h.plot(block=None)
plt.xlabel('Chromaticity (g)');
plt.grid(True)


The big strong green peak on the right is the green background, the other peak is the foreground object.  A threshold of 0.45 nicely separates the two pixel populations.

In [ ]:
mask = cc.plane('g') < 0.45
mask.disp()
print(mask)


The result is a logical image where True pixels indicate foreground pixels.

We want to apply this mask to a color image so we replicate this value across the RGB color channels

In [ ]:
mask3 = mask.colorize()
print(mask3)

which we see is a color image where the R, G, and B planes are the image above.

Now we can apply the mask to the foreground image and display it.

In [ ]:
(foreground * mask3).disp()

Happily, all the green pixels have now gone.

Next we load the background image, and clip and scale it to be the same size as the foreground image.

In [ ]:
background = Image.Read("road.png", dtype="float").samesize(foreground)

and then apply the inverse mask to it.  Note that in Python `True`=1 and `False`=0.

In [ ]:
(background * (1 - mask3)).disp();

Now, all that's left to do is to add the masked foreground image to the inverse masked background image

In [ ]:
(foreground * mask3  + background * (1 - mask3)).disp();

# Spatial operations

Now let's load a real greyscale image from a PNG file.  This particular image file is distributed with the Toolbox, but you can pass in the path to any image file you might have.  _If the Toolbox can't find the specified image it defaults to looking in the folder of images distributed with the Toolbox._

In [ ]:
mona = Image.Read("monalisa.png", mono=True)  # convert to monochrome
mona.disp();

For image smoothing it is preferable to use a kernel that is isotropic and symmetric such as a 2D Gaussian

$G(u,v) = \frac{1}{2\pi\sigma^2}e^{-\frac{u^2+v^2}{2\sigma^2}}$

  We will create a 2D Gaussian with $\sigma=10$

In [ ]:
from machinevisiontoolbox import Kernel

K = Kernel.Gauss(10)
print(K)

We can display the kernel as a 3-dimensional mesh.

In [ ]:
K.disp3d()

Now we can convolve the image with this kernel

In [ ]:
mona.convolve(K).disp();

We can do this more simply using the `smooth` method which accepts $\sigma$ and the half-kernel width.

In [ ]:
mona.smooth(2, 5).disp();

which shows how method chaining can be used to create simple and readable processing **pipelines** read from left to right.

image[mona] → smooth(2,5) → display

## Finding edges
We can use 2D filtering to find edges as well.  This convolution kernel will find vertical edges.  The intuition is that each row of this kernel subtracts the pixel to the left from the pixel to the right, which will give a positive value if the intensity is increasing left to right.

In [ ]:
kernel = Kernel.DoG(2, 5)

In [ ]:
penguins = Image.Read('penguins.png', grey=True, dtype='float')
penguins.disp();

In [ ]:
gradient_horizontal = penguins.convolve(kernel)
gradient_horizontal.disp(colormap='signed');                

The image is displayed with a color map that shows negative numbers as red and positive numbers as blue.  Zoom in on the outline of the "P" (use the second button from the right in the bottom toolbar) and you can see that the intensity goes up (blue) on the left side of the "P", from the grey background to the white paint. It goes down (red) on right of the stem, from the white paint to the gray background.

We can find the horizontal edges by finding vertical gradient, using the transpose of the kernel

In [ ]:
gradient_vertical = penguins.convolve(kernel.T)
gradient_vertical.disp(colormap='signed');    

# Image regions

We start by loading a simple binary image

In [ ]:
sharks = Image.Read("./shark2.png")
sharks.disp();

When we look at this we see two white objects, vaguely-shark shaped, against a black background.  

Finding the "objects" in the scene, grouping adjacent pixels of the same color. is a very classical computer vision algorithm. Such objects in a binary object are often called *blobs*.

Using this toolbox we simply write

In [ ]:
blobs = sharks.blobs();

The result is a feature object that describes the *blobs* present in the scene.
In this case there are


In [ ]:
len(blobs)

Each blob has a number of properties which are shown in the columns of the table.


In [ ]:
print(blobs)

We can put a box around the blobs and label them with their `id` number.

In [ ]:
sharks.disp(block=None)
blobs.plot_labelbox(color="yellow", linewidth=2)

This `blobs` object can be indexed or sliced just like a list.  Each element has a number of properties which were listed in the table earlier.  For example, its centroid (centre of mass) is

In [ ]:
blobs[0].centroid

its area in pixels

In [ ]:
blobs[0].area

and a bounding box

In [ ]:
blobs[0].bbox

where the first row is the u-axis range, and the second row is the v-axis range.  Alternatively we can consider the columns: the first column is the top-left coordinate and the second column is the bottom-right coordinate.

For each blob we can obtain the length of its perimeter

In [ ]:
blobs[0].perimeter_length

as well as the perimeter, as a set of points

In [ ]:
print(blobs[0].perimeter)

A simple but useful measure of "shape" is circularity, computed from area and perimeter.  It is 1 for a circle and 0 for a line.

In [ ]:
blobs[0].circularity

These properties can also be computed on a list of blob objects, and the result is an array or list. For example

In [ ]:
blobs.area

In [ ]:
blobs.centroid

The blob objects also support some graphical operations.
which depicts and labels each blob.  We also marked the centroids.

In [ ]:
sharks.disp(block=None)
blobs.plot_box(color="yellow", linewidth=2)
blobs.plot_centroid();

## Hierarchical blobs

Now we will load a more complex image that has blobs with holes that contain blobs with holes...

In [ ]:
multi = Image.Read("multiblobs.png", grey=True)
multi.disp();

In [ ]:
blobs = multi.blobs()
len(blobs)

and display the parameters as a table

In [ ]:
print(blobs)

The parent of blob 2 is

In [ ]:
blobs[2].parent

And the children of blob 2 are

In [ ]:
blobs[1].children

The obvious representation is a tree

In [ ]:
from IPython.display import Markdown, display

# Render dynamically via Jupyter's built-in Mermaid engine
if not globals().get("NBCONVERT", False):
    display(Markdown(blobs.graph(format="mermaid_fenced")))
    print()

The final thing we will do is to create a label image, a pixel classification, and display it

In [ ]:
labels = blobs.label_image()
labels.disp(
    colormap="viridis",
    ncolors=10,
    colorbar=dict(shrink=0.8, aspect=20 * 0.8),
    block=True,
);

Where the value of each pixel is the `id` of the blob it belongs to.  By drifting the cursor over the image you can see which pixels belong to say blob #6.  The image uses a colorful colormap to make it easy to see the different label areas.

# Camera modeling

The toolbox provides classes to model various types of cameras:

* `CentralCamera` a "standard" perspective camera like your eye or a digital camera, lines always map to lines
* `FisheyeCamera` a wide-angle camera, where lines don't map to lines
* `CatadoptricCamera` a panoramic camera with a lense and mirror, where lines don't map to lines
* `SphericalCamera` a non-implementable ideal camera with spherical field of view

We'll create a model of a regular camera.

First, define some parameters of our camera

In [ ]:
f = 8*1e-3     # focal length in metres
rho = 10*1e-6  # pixel side length in metres
u0 = 500       # principal point, horizontal coordinate
v0 = 500       # principal point, vertical coordinate

In [ ]:
from machinevisiontoolbox import CentralCamera

camera = CentralCamera(f=f, rho=rho, pp=(u0, v0), imagesize=1000)
print(camera)

We can very conveniently project points to the image plane

In [ ]:
P = [1,1,5]
camera.project_point(P)


or plot them on the camera's image plane

In [ ]:
camera.plot_point(P)

We can project the same point, but this time with the camera moved 0.5m in the x-direction

In [ ]:
from spatialmath import SE3

camera.plot_point(P, pose=SE3(0.5, 0, 0))

and we see that the u-coordinate has decreased.  If we look out along the camera's principal axis then moving the camera to the right causes the image plane point to move to the left.  Note that the vertical coordinate hasn't changed -- as expected.

We can extract the intrinsic matrix

In [ ]:
camera.K

and the camera matrix

In [ ]:
camera.C()

and also the camera matrix for the case where the camera is moved

# Image motion & Jacobians

In [ ]:
camera = CentralCamera.Default(name='')

P = [1,1,5]

p = camera.project_point(P)
print(p)

J = camera.visjac_p(p, depth=5)
print(J)

J @ [0, 0, 0, 0, 0, 0.1]

In [ ]:
# Tz
camera.flowfield( [0, 0, 1, 0, 0, 0] )


# Homographies

## Homography example 1: camera field of view

We define a central perspective camera (see camera.ipynb for more details on this), that is position up high, looking obliquely downward at the ground

In [ ]:
camera = CentralCamera(f=0.012, rho=10e-6, imagesize=1000, 
        pose=SE3(0, 0, 8) * SE3.Rx(-2.8))

And we can plot the camera in the 3D world

In [ ]:
ax = camera.plot(scale=2, shape='camera', color='k', frame=True)
ax.set_xlim(-8, 12)
ax.set_ylim(-10, 10)
ax.set_zlim(0, 10)

A shape on the ground plane is defined by a set of 2D coordinates

In [ ]:
P = np.column_stack([[-1, 1], [-1, 2], [ 2,2], [2, 1]])
P

To obtain the coordinates of the points in 3D, we augment each column with a zero, since the ground plane is defined by $z=0$

In [ ]:
P0 = np.vstack([P, np.zeros((4,))])
P0

Now we can project the 3D ground plane points onto the image plane

In [ ]:
camera.project_point(P0)

The homography is computed from the camera matrix by deleting column two (the z column)

In [ ]:
H = np.delete(camera.C(), 2, axis=1)
H

We can use this matrix to directly compute the image plane points, by transforming the homogeneous ground plane points

In [ ]:
from spatialmath.base import homtrans

homtrans(H, P)

which first converts `P` to homogeneous form, performs the multiplication, then converts the resulting homogeneous coordinates to Euclidean.


H is square and of full rank, so it is invertible. This means that we can perform the inverse mapping, from the image plane
to the ground plane.

The camera has a 1000 x 1000 image plane so the coordinates of its corners are


In [ ]:
p = np.column_stack([[0, 0], [0, 1000], [1000, 1000], [1000, 0]])
p

and on the ground plane these are the points

In [ ]:
from spatialmath.base import homtrans


Pi = homtrans(np.linalg.inv(H), p)
Pi

Now we can overlay the corners of the camera's field of view onto the "world view" of the imaging setup that we showed earlier

In [ ]:
ax = camera.plot(scale=2, shape='camera', color='k', frame=True)
k = [0, 1, 2, 3, 0]
ax.plot(Pi[0, k], Pi[1, k], np.zeros(5), 'b--')


Scroll back up to the previous figure to see the blue dashed line representing the field of view.

## Homography example 2: Perspective rectification

In [ ]:
im = Image.Read('notre-dame.png')
print(im)
p1 =   np.array([
    [44.1364,  377.0654], 
    [94.0065,  152.7850],
    [537.8506,  163.4019],
    [611.8247,  366.4486]
]).T

mn = p1.min(axis=1)
mx = p1.max(axis=1)
p2 = np.array([
    [mn[0], mn[0], mx[0], mx[0]],
    [mx[1], mn[1], mn[1], mx[1]]
])

H, _ = CentralCamera.points2H(p1, p2, method='leastsquares')

warped = im.warp_perspective(H)
warped.disp(grid=True)


# Stereo vision

In [ ]:
L = Image.Read('rocks2-l.png', reduce=2)
R = Image.Read('rocks2-r.png', reduce=2)

L.stdisp(R)

In [ ]:
disparity, sim, DSI = L.stereo_simple(R, 3, [40, 90])

disparity.disp(grid=True, badcolor='red', colorbar=dict(shrink=0.92, aspect=20*0.92, label='Disparity (pixels)'))

In [ ]:
disparity_refined, A = Image.DSI_refine(DSI)
disparity_refined.disp()

# SIFT features

## Finding correspondences between images

First we will load two different views of the same scene

In [ ]:
im1 = Image.Read('eiffel-1.png')
im2 = Image.Read('eiffel-2.png')
Image.Hstack((im1,im2)).disp()


and then use SIFT to find corresponding features in the two images

In [ ]:
f1 = im1.SIFT()
f2 = im2.SIFT()

In [ ]:
len(f1), len(f2)

Then we create a "*match*" object and give it the two sets of features.  It finds the best matches and for each match computes a `distance` which is a measure of dissimilarity.  We sort the matches into decreasing similarity

In [ ]:
matches = f1.match(f2)
matches

then plot the best 100 matches.  The `plot` method renders the matches onto the original images and overlays lines that connect corresponding points

In [ ]:
matches[:100].plot('y', alpha=0.6, block=None)

In [ ]:
F, resid = matches.estimate(CentralCamera.points2F, method="ransac", confidence=0.99, seed=0)

In [ ]:
print(matches)
print(matches[:10].list())

In [ ]:
matches.inliers.subset(100).plot(color="g")

In [ ]:
matches.outliers.subset(100).plot(color="r")

In [ ]:
plt.clf()
camera = CentralCamera(name="view1")
camera.disp(im1)

camera.plot_epiline(F.T, matches.inliers.subset(20).p2, color="black")

# Fiducial markers (April tags, ArUco tags)

In [ ]:
scene = Image.Read("lab-scene.png", rgb=False)
scene.disp();
print(scene)

We are attempting to determine the 3D pose of the arUco markers based on 2D image information, so we need to know some parameters of the imaging geometry.  We create a model of a central projection camera, using as many of the parameters as we know:
-  the focal length, in this case 4.25mm
-  the image size
- the principal point, where the optical axis passes through the image plane
- the pixel size, in this case 1.4μm

From the camera model we can derive the intrinsic parameter matrix

In [ ]:
camera = CentralCamera(f=4.25e-3, imagesize=(4032, 3024), pp=(2016, 1512), rho=1.4e-6)
camera.K

There are many different arUco marker families, here we are using `4x4_1000` which is a $4 \times 4$ grid of squares that can encode numbers from 0 to 9999.  We pass in the marker family, the camera intrinsic parameter matrix, and the side length of the marker.  This last parameter is important, because with perspective projection we cannot tell the difference between some large and distance or small and close.  Knowing the size help us estimate distance.

In [ ]:
markers = scene.fiducial("4x4_1000", K=camera.K, side=67e-3);

The return is a list of Marker objects, each has the id of the marker, and the coordinates of four corners which is enough to estimate the orientation of the planar marker in 3D , which is shown above.  We pick element 2 of the marker list  

In [ ]:
marker = markers[3]
marker.id

which is the marker with `id` of 5, the marker being held by the teddy bear.  The corners of this particular fiducial are

In [ ]:
marker.corners

and we can display them on the original image (zoomed in)

In [ ]:
import matplotlib.pyplot as plt

scene.disp(block=None)
plt.plot(marker.corners[0,:], marker.corners[1,:], 'or')
plt.xlim(800, 1200)
plt.ylim(1200, 800)

The pose of the fiducial, with respect to the camera, as an SE(3) matrix is

In [ ]:
# marker.pose  # not in JupyterLite

Finally, we can render a coordinate frame associated with the pose of each fiducial, into the original image.  The fiducial's z-axis is normal to its plane.

In [ ]:
# for marker in markers:  # not in JupyterLite
#     marker.draw(scene, length=0.10, thick=20)
# scene.disp();

# Package connectivity

![MVTB ecosystem](support/figs/package-connections.png)

In [ ]:
from machinevisiontoolbox import VideoFile

frames = VideoFile("traffic_sequence.mp4")
print("contains", len(frames))


In [ ]:
for i, frame in enumerate(frames):
    frame.disp()
    if i >= 4:
        break

In [ ]:
# frames.disp(fps=10)

In [ ]:
from machinevisiontoolbox import FileCollection

frames = FileCollection("calibration/*.jpg")
print("contains", len(frames))
frames[3].disp();

In [ ]:
if not globals().get("NBCONVERT", False):
    from machinevisiontoolbox import WebCam

    WebCam("https://uk.jokkmokk.jp/photo/nr4/latest.jpg").grab().disp()

In [ ]:
from machinevisiontoolbox import EarthView

# world = EarthView()
# world.grab(-27.475722, 153.0285, 17).disp()

In [ ]:
# frames = FileArchive("calibration.zip", pattern="*.png")

In [ ]:
# frames = ROSBag("race_1.bag")